In [2]:
pip install yt-dlp opencv-python easyocr openai

Note: you may need to restart the kernel to use updated packages.


In [28]:
import cv2
import yt_dlp
import easyocr
import numpy as np
import os
import re

# =========================
# 🔥 OCR 설정
# =========================
reader = easyocr.Reader(['ko','en'], gpu=False)

class ShortsOCRAnalyzer:
    def __init__(self, download_path="downloads"):
        self.download_path = download_path
        os.makedirs(download_path, exist_ok=True)

    # =========================
    # 1️⃣ 영상 다운로드
    # =========================
    def download_video(self, url):
        print("1️⃣ 다운로드 중...")

        ydl_opts = {
            'format': 'mp4',
            'outtmpl': f'{self.download_path}/shorts.%(ext)s',
            'quiet': True
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        return f"{self.download_path}/shorts.mp4"

    # =========================
    # 2️⃣ OCR 추출 (중앙 자막 대응)
    # =========================
    def extract_ocr(self, video_path):
        print("2️⃣ OCR 추출 중...")

        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)

        texts = []
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            # 🔥 0.3초 간격
            if frame_count % int(fps * 0.3) == 0:

                h, w = frame.shape[:2]

                # 🔥 중앙 60% 크롭
                top = int(h * 0.2)
                bottom = int(h * 0.8)
                crop = frame[top:bottom, 0:w]

                gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
                gray = cv2.equalizeHist(gray)

                # 샤프닝
                kernel = np.array([[0,-1,0],
                                   [-1,5,-1],
                                   [0,-1,0]])
                sharp = cv2.filter2D(gray, -1, kernel)

                # 🔥 Adaptive Threshold
                thresh = cv2.adaptiveThreshold(
                    sharp,
                    255,
                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                    cv2.THRESH_BINARY,
                    11,
                    2
                )

                result = reader.readtext(
                    thresh,
                    detail=0,
                    paragraph=False   # 🔥 중요
                )

                if result:
                    texts.extend(result)

            frame_count += 1

        cap.release()

        return " ".join(texts)

    # =========================
    # 3️⃣ 텍스트 정제
    # =========================
    def clean_text(self, text):

        # 숫자 제거
        text = re.sub(r'\d+', '', text)

        # 한글/영문만 남기기
        text = re.sub(r'[^가-힣a-zA-Z\s]', '', text)

        # 공백 정리
        text = re.sub(r'\s+', ' ', text)

        # 짧은 단어 제거
        words = text.split()
        words = [w for w in words if len(w) > 1]

        return " ".join(words).strip()

    # =========================
    # 4️⃣ 간단 규칙 판별
    # =========================
    def rule_check(self, text):

        if len(text) < 20:
            return {"score": 0, "reason": "텍스트 데이터 부족"}

        score = 0
        reason = []

        words = text.split()
        unique_ratio = len(set(words)) / len(words)

        if unique_ratio < 0.4:
            score += 2
            reason.append("반복 텍스트 많음")

        clickbait_words = ["충격", "절대", "무조건", "반드시", "몰랐던"]
        if any(word in text for word in clickbait_words):
            score += 1
            reason.append("어그로 키워드 포함")

        if max(words.count(w) for w in set(words)) > 5:
            score += 1
            reason.append("같은 단어 반복")

        return {
            "score": score,
            "reason": ", ".join(reason) if reason else "정상 패턴"
        }

    # =========================
    # 5️⃣ 실행
    # =========================
    def run(self, url):
        video_path = self.download_video(url)
        raw_text = self.extract_ocr(video_path)

        print("\n[원본 OCR 텍스트]")
        print(raw_text)

        cleaned = self.clean_text(raw_text)

        print("\n[정제 텍스트]")
        print(cleaned)

        result = self.rule_check(cleaned)

        print("\n[최종 판별]")
        print(result)


# =========================
# 🔥 실행
# =========================
if __name__ == "__main__":
    url = input("쇼츠 URL 입력: ")
    analyzer = ShortsOCRAnalyzer()
    analyzer.run(url)

Using CPU. Note: This module is much faster with a GPU.


1️⃣ 다운로드 중...


2️⃣ OCR 추출 중...              


c:\Users\hu\anaconda3\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



[원본 OCR 텍스트]
음료에 음료예 넣으려더 넣으리런 이퍼곤 사실흘 사실흘 @제프m @제컨m [제 떼$ 팬 폐미 떼판 @깥 올완렉제마 3   i 뿐 (탤잔데 슈 관 발리 훨록데 찌_기 탤텔레렉 스닭 [텔래늘 -이킬깨러교 원레늘 깨러맨 박울베때날 [려터 탁-메뼈달 때터 핑봉튼 탁기메j발 빼레터계름구 태기때뼈발 탈기패팬m 폐러요기튼 손녀분 흔녀분 천재 아프까요? 천재 아별까요? 전재 아필i오3

[정제 텍스트]
음료에 음료예 넣으려더 넣으리런 이퍼곤 사실흘 사실흘 제프m 제컨m 폐미 떼판 올완렉제마 탤잔데 발리 훨록데 찌기 탤텔레렉 스닭 텔래늘 이킬깨러교 원레늘 깨러맨 박울베때날 려터 탁메뼈달 때터 핑봉튼 탁기메j발 빼레터계름구 태기때뼈발 탈기패팬m 폐러요기튼 손녀분 흔녀분 천재 아프까요 천재 아별까요 전재 아필i오

[최종 판별]
{'score': 0, 'reason': '정상 패턴'}
